In [26]:
# importanto a biblioteca duckdb para criar o banco de dados em memória
import duckdb

In [27]:
#criando a conexão com o banco de dados duckdb
con =duckdb.connect(database='dados_duckdb.db', read_only=False)

In [28]:
# ceriando a tabela bronze_Z0019 no banco de dados
df = con.execute('SELECT * FROM bronze_Z0019').fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-30 10:09:05.900232
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-30 10:09:05.900232
2,10003,PREGO,BT10,100,500,z0019_1.csv,2026-07-30 10:09:05.900232
3,10004,SERRA,BT50,100,200,z0019_1.csv,2026-07-30 10:09:05.900232
4,10005,MACHADO,BT50,100,100,z0019_1.csv,2026-07-30 10:09:05.900232
5,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-30 10:16:19.439626
6,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-30 10:16:19.439626
7,10003,PREGO,BT10,100,600,z0019_2.csv,2026-07-30 10:16:19.439626


In [29]:
# crieando um dataframe com os dados mais recentes da tabela bronze_Z0019, considerando a coluna data_ingestao
df = con.execute("""
                    select * from (
                    select * , row_number() over (partition by NATBR order by data_ingestao desc) AS row
                    from bronze_Z0019 
                    where data_ingestao >= '2026-07-30') where row = 1
                    """).fetchdf()
df.head(10)


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-30 10:09:05.900232,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-30 10:09:05.900232,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-30 10:16:19.439626,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-30 10:16:19.439626,1
4,10003,PREGO,BT10,100,600,z0019_2.csv,2026-07-30 10:16:19.439626,1


In [30]:
# apagando as colunas que não serão utilizadas no dataframe final com o comando drop do pandas
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])

df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,600


In [46]:
# renomeando as colunas do dataframe final para padronizar com o banco de dados
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_preco'})


df_final.head(10)


,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,600


In [52]:
df_final.dtypes


id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [60]:
#MODULO 04 VIDEOAULA 1.4G
df2 = df_final
df2 = df2.astype({'id': 'int32', 'nm_produto': 'string', 'id_categoria': 'string', 'id_fornecedor': 'int32', 'vl_preco': 'float32'})

df2.head(10)


,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,600.0


In [61]:
df2.dtypes

id                 int32
nm_produto        string
id_categoria      string
id_fornecedor      int32
vl_preco         float32
dtype: object

In [ ]:
# passando a tipagem python para o banco de dados duckdb

df2 = df2.astype(
    {
        'id': int, 
        'nm_produto':str, 
        'id_categoria': str, 
        'id_fornecedor': str, 
        'vl_preco': float
    }
)

df2.dtypes

id                 int64
nm_produto           str
id_categoria         str
id_fornecedor        str
vl_preco         float64
dtype: object

In [ ]:
# para armanezar os dados, vamos precisar criar uma tabela no banco de dados duckdb, para isso vamos utilizar o comando CREATE TABLE do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.

con.execute('CREATE TABLE IF NOT EXISTS produtos (id INTEGER, nm_produto TEXT, id_categoria TEXT, id_fornecedor BIGINT, vl_preco FLOAT)')

In [ ]:
# visualizando os tipos de dados das colunas do dataframe df2

df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,600.0


In [ ]:
# apos a tabela produtos criada, vamos visualizar os dados que estão na tabela produtos, para isso vamos utilizar o comando SELECT do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.
# retorna vazia pois acabamos de criar a tabela produtos, e ainda não inserimos nenhum dado nela.
df_resultado = con.execute('SELECT * FROM produtos').fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco


In [70]:
# agora vamos inserir dados na tabela produtos, para isso vamos utilizar o comando INSERT INTO do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.

con.execute('INSERT INTO produtos SELECT * FROM df2')

In [71]:
# agoar consulto os dados inseridos na tabela produtos, para isso vamos utilizar o comando SELECT do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.
df_resultado = con.execute('SELECT * FROM produtos').fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,600.0


In [72]:
# fechar a conexão com o banco de dados duckdb
con.close()


# levar os dados camada gold - video 1.4_H

In [73]:
#cria a conexão com o banco de dados duckdb
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [74]:
# selecionar os produtos da tabela produtos
df_resultado = con.execute('SELECT * FROM produtos').fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,600.0


In [75]:
# criar dimensao na camada gold
# vamos deletar os registors de categora e fornecedor.

df2 = df_resultado.drop(columns=['id_categoria', 'id_fornecedor'])
df2.head(10)

,id,nm_produto,vl_preco
0,10001,PARAFUSO,100.0
1,10002,MARTELO,1500.0
2,10004,SERRA,200.0
3,10005,MACHADO,100.0
4,10003,PREGO,600.0


In [76]:
# criar tabela de dimensao produtos_gold no banco de dados duckdb
con.execute('CREATE TABLE IF NOT EXISTS dim_produtos (id_produto BIGINT, NM_PRODUTO TEXT, VL_PRODUTO FLOAT)')

In [77]:
# consultar a tabela dim_produtos no banco de dados duckdb
df_dim_produtos = con.execute('SELECT * FROM dim_produtos').fetchdf()  
df_dim_produtos.head(10)

,id_produto,NM_PRODUTO,VL_PRODUTO


In [99]:
# excluir tabela de dimensao produtos_gold no banco de dados duckdb e criar os campos em minnusculo
con.execute('DROP TABLE IF EXISTS dim_produtos')
con.execute('CREATE TABLE IF NOT EXISTS dim_produtos (id_produto BIGINT, nm_produto TEXT, vl_produto FLOAT)')



In [101]:
#consultar novamente a tabela dim_produtos no banco de dados duckdb
df_dim_produtos = con.execute('SELECT * FROM dim_produtos').fetchdf()
df_dim_produtos.head(10)

df2.head

<bound method NDFrame.head of       id nm_produto  vl_preco
0  10001   PARAFUSO     100.0
1  10002    MARTELO    1500.0
2  10004      SERRA     200.0
3  10005    MACHADO     100.0
4  10003      PREGO     600.0>

In [102]:
# inserir os dados na tabela dim_produtos no banco de dados duckdb, para isso vamos utilizar o comando INSERT INTO do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.
con.execute('INSERT into dim_produtos SELECT * FROM df2')



In [103]:
# vamos criar o df_dim_produtos e visualizar os dados inseridos na tabela dim_produtos, para isso vamos utilizar o comando SELECT do SQL, e para isso vamos utilizar o método execute da conexão com o banco de dados duckdb.
df_dim_produtos = con.execute('SELECT * FROM dim_produtos').fetchdf()
df_dim_produtos.head(10)

,id_produto,nm_produto,vl_produto
0,10001,PARAFUSO,100.0
1,10002,MARTELO,1500.0
2,10004,SERRA,200.0
3,10005,MACHADO,100.0
4,10003,PREGO,600.0


In [104]:
con.close

<bound method pybind11_detail_function_record_v1_msvc_mt_mscver1944.close of <_duckdb.DuckDBPyConnection object at 0x0000028B1B6176F0>>